# Lowrox — MLOps and the feature store

Chapter 7, built on the Lowrox lakehouse: **Staging → Feature Store → Model →
Registry → Serving → Monitoring → Dashboard**, with CI/CD around all of it.

| Chapter 7 | Where it is here |
| --- | --- |
| DevOps vs MLOps — CI validates data, schemas and models (slide 4) | Part 5, four CI suites |
| The eight data science steps (slides 5–6) | Parts 3–8 |
| MLOps level 0 / 1 / 2 (slides 7–12) | Part 1 for the assessment, Part 6 for the six stages |
| Model drift — data, label, concept (slides 13–16) | Part 8, each one simulated |
| Edge cases, feedback loops, missing-as-a-feature (slides 17–19) | Part 8 |
| Feature stores — index, time index, features (slides 20–24) | Part 3 |
| Feature engineering (slides 25–38) | Part 4 |

The ML problem is deliberately the one slide 24 uses: **churn**, with a label
table of `(user_id, ts, is_churn)` enriched from several feature sets.

Three things in this notebook went wrong before they went right, and the
wrong versions are described rather than hidden — the first model scored
ROC-AUC 0.995 and was worthless, and finding out why is most of what this
chapter is about.

## Part 0 — Setup

Runs on Colab with nothing installed, and needs **no database**. If the
lakehouse from the ETL notebook is present with enough members it is used; if
not, the same calibrated generator produces a larger synthetic set straight
into the Silver zone.

Why that matters: the Supabase project holds 50 members. Nine monthly
snapshots over 50 members is a few hundred label rows at a 9% churn rate, so
the test split would contain a handful of churn events and every metric would
be noise. The pipeline is identical at either size; only whether the numbers
mean anything changes.

In [1]:
import sys, os, subprocess, glob, zipfile
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "deltalake==1.6.6", "pyarrow", "pandas", "scikit-learn",
                    "scipy", "matplotlib"], check=True)

GITHUB_REPO = "https://github.com/daruntoey/lowrox_etl.git"
CLONE_INTO  = "/content/lowrox_etl"

def _git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, capture_output=True, text=True)

if IN_COLAB and GITHUB_REPO:
    if os.path.isdir(os.path.join(CLONE_INTO, ".git")):
        # "Restart session" does not delete /content, so an old checkout
        # survives every restart. Update rather than skip.
        _git("fetch", "--depth", "1", "origin", cwd=CLONE_INTO)
        before = _git("rev-parse", "HEAD", cwd=CLONE_INTO).stdout.strip()[:7]
        r = _git("reset", "--hard", "origin/HEAD", cwd=CLONE_INTO)
        if r.returncode != 0:
            r = _git("reset", "--hard", "origin/main", cwd=CLONE_INTO)
        after = _git("rev-parse", "HEAD", cwd=CLONE_INTO).stdout.strip()[:7]
        print(f"checkout {'updated ' + before + ' -> ' + after if before != after else 'already current (' + after + ')'}")
    else:
        r = _git("clone", "--depth", "1", GITHUB_REPO, CLONE_INTO)
        print(f"cloned into {CLONE_INTO}" if r.returncode == 0 else r.stderr.strip())

def find_project() -> str:
    for cand in (CLONE_INTO, "/content/lowrox_etl", os.getcwd(),
                 os.path.join(os.getcwd(), "lowrox_etl")):
        if os.path.isfile(os.path.join(cand, "mlops", "feature_store.py")):
            return os.path.abspath(cand)
    for z in glob.glob("/content/*.zip") + glob.glob("*.zip"):
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content" if IN_COLAB else ".")
    for hit in (glob.glob("/content/**/mlops/feature_store.py", recursive=True)
                + glob.glob("./**/mlops/feature_store.py", recursive=True)):
        return os.path.abspath(os.path.dirname(os.path.dirname(hit)))
    raise RuntimeError("could not find the mlops/ package — upload lowrox_etl.zip "
                       "with the folder icon and re-run this cell")

PROJECT_DIR = os.environ.get("LOWROX_PROJECT_DIR") or find_project()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.environ.setdefault("LOWROX_LAKE_ROOT",
                      "/content/lowrox_lake" if IN_COLAB else os.path.join(PROJECT_DIR, "lowrox_lake"))
os.environ.setdefault("LOWROX_MLOPS_ROOT",
                      "/content/mlops_store" if IN_COLAB else os.path.join(PROJECT_DIR, "mlops_store"))

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200, "display.max_columns", 50)

from mlops import (config, demo_data, feature_store as FS, features as F,
                   training as T, registry as REG, ci as CI, serving as SRV,
                   monitoring as MON, orchestrator as ORC, simulate as SIM,
                   dashboard as DASH)
from mlops.sources import load_sources

paths = config.wire()
print(f"mlops package {__import__('mlops').__version__}")
print(f"project : {PROJECT_DIR}")
print(f"lake    : {config.LAKE_ROOT}")
for k, v in paths.items():
    print(f"  {k:16} {v}")

mlops package 2026.11.13
project : /home/claude/lowrox_etl
lake    : /home/claude/lowrox_lake_nb
  feature_store    /home/claude/mlops_store_nb/feature_store
  model_registry   /home/claude/mlops_store_nb/model_registry
  ml_metadata      /home/claude/mlops_store_nb/ml_metadata
  predictions      /home/claude/mlops_store_nb/predictions
  dashboard        /home/claude/mlops_store_nb/lowrox_dashboard.html


## Part 1 — Where this pipeline sits: MLOps level 0, 1 or 2

Slides 7, 9 and 11 describe three maturity levels. Naming the level honestly
is more useful than claiming the highest one, because the gap is the roadmap.

In [2]:
maturity = pd.DataFrame([
 {"capability":"ML pipeline is code, not notebook cells","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — mlops/ package"},
 {"capability":"Feature store between raw data and models","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — Part 3"},
 {"capability":"ML metadata store records every run","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — run_log.jsonl"},
 {"capability":"Automated trigger for retraining","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — schedule, new data, drift"},
 {"capability":"Continuous training in production","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — stage 4"},
 {"capability":"CI validates data, schemas AND models","level_0":"no","level_1":"manual","level_2":"yes","lowrox":"yes — Part 5"},
 {"capability":"CI/CD deploys the PIPELINE automatically","level_0":"no","level_1":"no","level_2":"yes","lowrox":"yes — .github/workflows"},
 {"capability":"Model promotion gated on beating production","level_0":"no","level_1":"no","level_2":"yes","lowrox":"yes — stage 5"},
 {"capability":"Active performance monitoring in production","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"yes — Part 8"},
 {"capability":"Live traffic feeding the monitors","level_0":"no","level_1":"yes","level_2":"yes","lowrox":"NO — no real users yet"},
])
print(maturity.to_string(index=False))
print("""
Assessment: the pipeline implements MLOps level 2. The one honest gap is the
last row — there are no real users, so monitoring runs against held-out and
simulated data rather than production traffic. Everything upstream of that is
in place, which is the useful thing to be able to say in the report.""")

                                 capability level_0 level_1 level_2                          lowrox
    ML pipeline is code, not notebook cells      no     yes     yes            yes — mlops/ package
  Feature store between raw data and models      no     yes     yes                    yes — Part 3
        ML metadata store records every run      no     yes     yes             yes — run_log.jsonl
           Automated trigger for retraining      no     yes     yes yes — schedule, new data, drift
          Continuous training in production      no     yes     yes                   yes — stage 4
      CI validates data, schemas AND models      no  manual     yes                    yes — Part 5
   CI/CD deploys the PIPELINE automatically      no      no     yes         yes — .github/workflows
Model promotion gated on beating production      no      no     yes                   yes — stage 5
Active performance monitoring in production      no     yes     yes                    yes — Part 8


## Part 2 — Staging: the data the model will learn from

Features are built from **Silver**, never from the source database. Silver is
the trust boundary established in the ETL pipeline: a row only reaches it after
passing the quality gate. A feature computed from a row that failed that gate
would be a feature the warehouse itself does not believe.

In [3]:
info = demo_data.ensure_training_data(config.LAKE_ROOT, project_dir=PROJECT_DIR,
                                      min_users=1000, scale=40)
snaps = F.snapshot_dates(config.SNAPSHOT_START, config.SNAPSHOT_END)
src = load_sources(config.LAKE_ROOT, snaps)
print("\nSILVER TABLES AVAILABLE TO THE FEATURE PIPELINES")
print(pd.DataFrame([{"table": k, "rows": len(v)} for k, v in src.items()
                    if k != "snapshots"]).to_string(index=False))
print(f"\nsnapshot grid: {[str(d.date()) for d in snaps]}")

Silver already holds 2,000 members — using it as is.

SILVER TABLES AVAILABLE TO THE FEATURE PIPELINES
                 table  rows
              ms_users  2000
  ms_user_demographics  2000
 ms_subscription_plans     6
txn_user_subscriptions  2000
 txn_workout_summaries  2000
     txn_notifications  2000
txn_event_participants  2000
    txn_group_comments  2000
          txn_payments  2000
            txn_events  2000
 ms_device_connections  2000

snapshot grid: ['2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31', '2026-09-30', '2026-10-31']


## Part 3 — The feature store (slides 20–24)

Slide 23 gives the shape of a feature set: **index + time index + features**.
Slide 22 gives the reason it exists: the same features must serve training and
inference, or the two silently disagree.

Everything here is computed on a **monthly snapshot grid**. For each member and
each month end, the features describe only what had happened *strictly before*
that month end. That is what gives each feature set a real time index, and it
is what makes the point-in-time join something other than decoration.

In [4]:
for fs in F.ALL_FEATURE_SETS:
    print(f"{fs.name}")
    print(f"   index={fs.entity}  time_index={fs.timestamp}  "
          f"{len(fs.features)} features")
    print(f"   {fs.description}")
print("\nMATERIALISING — one computation, written to BOTH stores")
for fs in F.ALL_FEATURE_SETS:
    r = FS.materialise(fs, src, run_id="nb")
    print(f"   {r['feature_set']:28} offline {r['offline_rows']:>6} rows "
          f"(full history)   online {r['online_rows']:>5} rows (latest per member)")
print()
print(FS.REGISTRY.catalogue().to_string(index=False))

user_profile_features
   index=user_id  time_index=ts  6 features
   Who the member is, as at each month end: tenure, age band, tier, referral origin, hashed home area.
workout_activity_features
   index=user_id  time_index=ts  21 features
   Training behaviour in 30 and 90 day windows as at each month end, with lags, recency, pace trend and cyclical time.
engagement_features
   index=user_id  time_index=ts  6 features
   How the member responds to the product: notification opens, event participation, group comments.
subscription_features
   index=user_id  time_index=ts  8 features
   Commercial state of the membership: age, price, auto-renew, payment history and failures.

MATERIALISING — one computation, written to BOTH stores


   user_profile_features        offline  10542 rows (full history)   online  2000 rows (latest per member)


   workout_activity_features    offline  18000 rows (full history)   online  2000 rows (latest per member)


   engagement_features          offline  18000 rows (full history)   online  2000 rows (latest per member)


   subscription_features        offline   7931 rows (full history)   online  1993 rows (latest per member)

              feature_set   index time_index  n_features                                                              features  rows_offline   last_materialised       owner
    user_profile_features user_id         ts           6                  tenure_days, log_tenure_days, age_band, gender_code…         10542 2026-11-13T17:09:40 lowrox-data
workout_activity_features user_id         ts          21 sessions_lifetime, log_sessions_lifetime, sessions_30d, sessions_90d…         18000 2026-11-13T17:09:41 lowrox-data
      engagement_features user_id         ts           6         notifications_30d, opens_30d, open_rate_30d, events_lifetime…         18000 2026-11-13T17:09:41 lowrox-data
    subscription_features user_id         ts           8   subscription_age_days, price_thb, plan_is_elite, payments_lifetime…          7931 2026-11-13T17:09:42 lowrox-data


### The label table and the point-in-time join

Slide 24's top table is `customer_churn | Model` — `(c_id, ts, is_churn)`. The
Lowrox equivalent is below. The label looks **forward** (did this member cancel
within 60 days of the snapshot?) while the features look only **backward**. That
asymmetry is the whole design.

In [5]:
labels = F.build_labels(src)
print("LABEL TABLE  —  slide 24's customer_churn | Model")
print(labels.head(5).to_string(index=False))
print(f"\n{len(labels):,} rows, churn rate {labels.is_churn.mean():.1%}")
print(labels.assign(month=labels.ts.dt.strftime("%Y-%m")).groupby("month").agg(
    members=("is_churn","size"), churned=("is_churn","sum"),
    rate=("is_churn","mean")).round(3).to_string())

LABEL TABLE  —  slide 24's customer_churn | Model
 user_id                        ts  is_churn
       6 2026-02-28 00:00:00+00:00         0
       7 2026-02-28 00:00:00+00:00         0
      56 2026-02-28 00:00:00+00:00         0
      60 2026-02-28 00:00:00+00:00         0
      68 2026-02-28 00:00:00+00:00         0

7,931 rows, churn rate 9.1%
         members  churned   rate
month                           
2026-02      113       16  0.142
2026-03      331       46  0.139
2026-04      496       66  0.133
2026-05      711       94  0.132
2026-06      897       89  0.099
2026-07     1069       90  0.084
2026-08     1242       96  0.077
2026-09     1418      114  0.080
2026-10     1654      109  0.066


In [6]:
# slide 24's bottom table: the label enriched from every feature set
training = FS.get_training_data(labels, F.ALL_FEATURE_SETS, verbose=True)
print("\nENRICHED TRAINING TABLE")
print(training[["user_id","ts","is_churn","tenure_days","sessions_30d",
                "days_since_last","open_rate_30d","failed_payments",
                "price_thb"]].head(5).to_string(index=False))

POINT-IN-TIME JOIN
              feature_set  features_added  rows_matched  match_rate
    user_profile_features               6          7931       100.0
workout_activity_features              21          7931       100.0
      engagement_features               6          7931       100.0
    subscription_features               8          7931       100.0

7931 label rows enriched with 41 features

ENRICHED TRAINING TABLE
 user_id                        ts  is_churn  tenure_days  sessions_30d  days_since_last  open_rate_30d  failed_payments  price_thb
       6 2026-02-28 00:00:00+00:00         0           34           0.0            999.0            0.0              0.0     8990.0
    1625 2026-02-28 00:00:00+00:00         0           32           0.0            999.0            0.0              0.0     8990.0
    1620 2026-02-28 00:00:00+00:00         0            6           0.0            999.0            0.0              0.0      899.0
    1619 2026-02-28 00:00:00+00:00         0 

The join is `merge_asof` with `direction="backward"`: for each label row it
takes the most recent feature row **at or before** that label's timestamp.

An ordinary join on `user_id` alone would attach today's feature values to a
label from six months ago. The model would be trained on information that did
not exist when the prediction would have been made, score beautifully offline,
and collapse in production. This is the single most common way a feature store
earns its keep — so it is worth testing rather than trusting.

In [7]:
print("LEAKAGE CHECK — walk sampled rows back through the offline history")
print(FS.assert_no_future_leakage(training, F.ALL_FEATURE_SETS))

LEAKAGE CHECK — walk sampled rows back through the offline history


{'rows_checked': 800, 'leaks_found': 0, 'verdict': 'no future data used'}


## Part 4 — Feature engineering (slides 25–38)

Every technique the chapter lists, and where it is used on Lowrox data.

In [8]:
techniques = pd.DataFrame([
 {"slide":26,"technique":"Missing data imputation (KNN)","where":"inside the model pipeline, k=5"},
 {"slide":27,"technique":"Binning","where":"age_band, pace_band"},
 {"slide":27,"technique":"One-hot encoding","where":"plan_is_elite"},
 {"slide":27,"technique":"Label encoding","where":"gender_code"},
 {"slide":27,"technique":"Hashing (high cardinality)","where":"area_hash — 50 training areas"},
 {"slide":28,"technique":"Aggregation / statistical","where":"sessions, km, pace over 30d and 90d"},
 {"slide":28,"technique":"Rolling and lag","where":"sessions_30d_lag1, sessions_delta, pace_trend"},
 {"slide":29,"technique":"Normalisation (z-score)","where":"StandardScaler in the pipeline"},
 {"slide":29,"technique":"Log transform","where":"log_tenure_days, log_km_lifetime, log_revenue"},
 {"slide":29,"technique":"Feature interactions","where":"consistency_x_volume"},
 {"slide":30,"technique":"Cyclical date/time","where":"month_sin, month_cos"},
 {"slide":38,"technique":"Domain-specific","where":"days_since_last, failed_payment_rate, is_social"},
])
print(techniques.to_string(index=False))
print(f"\n{sum(len(fs.features) for fs in F.ALL_FEATURE_SETS)} features across "
      f"{len(F.ALL_FEATURE_SETS)} feature sets")

 slide                     technique                                           where
    26 Missing data imputation (KNN)                  inside the model pipeline, k=5
    27                       Binning                             age_band, pace_band
    27              One-hot encoding                                   plan_is_elite
    27                Label encoding                                     gender_code
    27    Hashing (high cardinality)                   area_hash — 50 training areas
    28     Aggregation / statistical             sessions, km, pace over 30d and 90d
    28               Rolling and lag   sessions_30d_lag1, sessions_delta, pace_trend
    29       Normalisation (z-score)                  StandardScaler in the pipeline
    29                 Log transform   log_tenure_days, log_km_lifetime, log_revenue
    29          Feature interactions                            consistency_x_volume
    30            Cyclical date/time                            m

### The columns that are deliberately **not** features

This cost three rounds of debugging, and it is the most useful thing in the
notebook.

A point-in-time join can only be as honest as the columns it joins. Silver
holds the **current** value of a mutable column, not its history — so reading
one at a past snapshot silently returns a value from the future. Where the
application mutates that column as part of the very event being predicted, the
"feature" *is* the label.

Each time, the symptom was identical: ROC-AUC near 0.995 and one feature
holding most of the importance.

In [9]:
print("COLUMNS EXCLUDED, AND WHY")
for col, why in F.MUTABLE_CURRENT_STATE.items():
    print(f"  {col:42} {why}")
print("""
Safe, because immutable or append-only:
  created_at / started_at / birth_date     written once
  plan_id and the plan's tier and price    not rewritten on cancellation
  workouts, notifications, payments        event tables, append-only

The subtlety worth keeping: the point-in-time JOIN was correct every time.
The leak was in the feature DEFINITION. A column can only be a point-in-time
feature if its history exists; otherwise its present value is a message from
the future.""")

COLUMNS EXCLUDED, AND WHY
  ms_users.tier                              set to 'free' at cancellation — 490 of 490 cancelled subscriptions have tier='free'
  txn_user_subscriptions.auto_renew          cleared at cancellation — auto_renew=false matches cancelled 490/490
  txn_user_subscriptions.status              'cancelled' IS the label
  txn_user_subscriptions.cancelled_at        used to BUILD the label, never as a feature

Safe, because immutable or append-only:
  created_at / started_at / birth_date     written once
  plan_id and the plan's tier and price    not rewritten on cancellation
  workouts, notifications, payments        event tables, append-only

The subtlety worth keeping: the point-in-time JOIN was correct every time.
The leak was in the feature DEFINITION. A column can only be a point-in-time
feature if its history exists; otherwise its present value is a message from
the future.


## Part 5 — Training, evaluation, validation (slides 5–6) and CI (slide 4)

The eight data science steps, with two details that matter more than the
algorithm choice:

**The split is by time, not at random.** The data is a panel — the same member
appears at several month ends and their rows are not independent. A random
split puts September's row for a member in training and October's in test, and
the model scores well by recognising the member rather than by learning churn.

**Imputation and scaling live inside the pipeline.** Fitting an imputer on the
whole frame before splitting leaks test-set statistics into training, subtly
and with no error message.

In [10]:
result = T.train_pipeline(labels, F.ALL_FEATURE_SETS, algorithm="random_forest",
                          run_id="nb", verbose=True, register_model=False)

ALGORITHM random_forest
  rows 7931  features 37 of 41 usable
  dropped >50% missing: ['avg_pace_30d', 'avg_pace_90d', 'avg_hr_30d', 'best_pace']
  split by time  train ['2026-02-28', '2026-06-30']  val ['2026-07-31', '2026-08-31']  test ['2026-09-30', '2026-10-31']
  train 2548 / val 2311 / test 3072
  calibrated on the validation split (sigmoid): Brier 0.1376 -> 0.0714

  test  roc_auc 0.7892  pr_auc 0.1791  recall@top20% 0.4305
  base  roc_auc 0.5000  pr_auc 0.0726

  VALIDATION GATES
                                               gate  value  threshold  passed
                                roc_auc above floor 0.7892     0.6000    True
                          beats baseline on roc_auc 0.2892     0.0500    True
      better calibrated than baseline (lower Brier) 0.0632     0.0698    True
    roc_auc not implausibly high (leakage sentinel) 0.7892     0.9800    True
no single feature dominates (subscription_age_days) 0.1714     0.5000    True

  verdict: PASSED — registered as a ca

### Validation gates, including two leakage sentinels

Slide 6 asks that a model be *"confirmed to be adequate for deployment — that
its predictive performance is better than a certain baseline."* Three gates do
that comparison. Two more exist because of the failures above: **a model can be
rejected for being too good.** An implausible score is evidence about the data,
not about the model.

In [11]:
print(result["validation"]["checks"].to_string(index=False))
print("\nTOP FEATURES — importance spread across many features, as it should be")
print(result["importances"].head(12).to_string(index=False))

                                               gate  value  threshold  passed
                                roc_auc above floor 0.7892     0.6000    True
                          beats baseline on roc_auc 0.2892     0.0500    True
      better calibrated than baseline (lower Brier) 0.0632     0.0698    True
    roc_auc not implausibly high (leakage sentinel) 0.7892     0.9800    True
no single feature dominates (subscription_age_days) 0.1714     0.5000    True

TOP FEATURES — importance spread across many features, as it should be
              feature  importance
subscription_age_days    0.171448
          tenure_days    0.126635
      log_tenure_days    0.112242
            area_hash    0.057876
      failed_payments    0.036154
  failed_payment_rate    0.036151
             age_band    0.032002
            price_thb    0.027656
         was_referred    0.027079
      days_since_last    0.025757
    comments_lifetime    0.025547
          log_revenue    0.025102


### Continuous integration (slide 4)

> *"CI is no longer only about testing and validating code and components, but
> also testing and validating data, data schemas, and models."*

Four suites. A normal software CI has only the first.

In [12]:
modules = {"feature_store": FS, "features": F, "training": T,
           "registry": REG, "serving": SRV, "monitoring": MON}
ci_report = CI.run_ci(modules, result, F.ALL_FEATURE_SETS, verbose=True)

STAGE 2 — PIPELINE CONTINUOUS INTEGRATION

  CODE  6/6 passed
     [PASS] feature_store exposes its interface  — 3 symbols
     [PASS] features exposes its interface  — 3 symbols
     [PASS] training exposes its interface  — 4 symbols
     [PASS] registry exposes its interface  — 4 symbols
     [PASS] serving exposes its interface  — 3 symbols
     [PASS] monitoring exposes its interface  — 3 symbols

  DATA  6/6 passed
     [PASS] enough rows to train  — 7931 rows
     [PASS] label is not degenerate  — positive rate 9.08%
     [PASS] no duplicate (entity, timestamp) rows  — 0 duplicates
     [PASS] no entirely empty feature  — []
     [PASS] no infinities  — []
     [PASS] no feature almost perfectly correlated with the target  — []

  SCHEMA  8/8 passed
     [PASS] user_profile_features contributes all its features  — 6 features present
     [PASS] workout_activity_features contributes all its features  — 21 features present
     [PASS] engagement_features contributes all its feature

## Part 6 — CI/CD pipeline automation (slides 11–12)

Slide 12's six stages, run in order. The ML metadata store records each one, so
*"why is this model in production"* has an answer after the fact.

In [13]:
run_id = ORC.new_run_id()
print(f"run {run_id}\n")
s1 = ORC.stage_experiment(run_id, labels=labels, feature_sets=F.ALL_FEATURE_SETS,
        algorithms=["gradient_boosting","random_forest","logistic_regression"],
        train_fn=T.train_pipeline, verbose=True)

run mlrun_20261113T170946_5885



STAGE 1 — DEVELOPMENT AND EXPERIMENT
          algorithm  val_roc_auc  test_roc_auc  test_pr_auc  brier  gates_passed
logistic_regression       0.7188        0.7827       0.1769 0.0654          True
  gradient_boosting       0.7089        0.7492       0.1579 0.0655          True
      random_forest       0.7057        0.7892       0.1791 0.0632          True

chosen on validation ROC-AUC: logistic_regression


The winner is chosen on **validation**, never on test. The test set is spent
once; choosing on it turns it into a second validation set and the reported
metric stops being an estimate of production performance.

In [14]:
print("STAGE 2 — PIPELINE CI  (already run above)")
print(f"  {(ci_report['results'].result=='PASS').sum()}/{len(ci_report['results'])} checks passed\n")

from mlops import __version__ as pkg_version
ORC.stage_cd_pipeline(run_id, package_version=pkg_version)

print("\nSTAGE 4 — AUTOMATED TRIGGERING")
triggers = ORC.evaluate_triggers(last_trained=None,
    now=pd.Timestamp.now(tz="UTC").to_pydatetime(),
    new_rows_since=len(labels), monitoring=None)
print(pd.DataFrame([{"trigger":t.kind,"fired":t.fired,"reason":t.reason}
                    for t in triggers]).to_string(index=False))

r4 = ORC.stage_continuous_training(run_id, labels=labels,
        feature_sets=F.ALL_FEATURE_SETS, algorithm=s1["best_algorithm"],
        train_fn=T.train_pipeline, verbose=False)
print(f"\n  trained {r4['model_version'].version}  "
      f"roc_auc={r4['metrics_test']['roc_auc']:.4f}  stage={r4['model_version'].stage}")

STAGE 2 — PIPELINE CI  (already run above)
  25/25 checks passed

STAGE 3 — PIPELINE CONTINUOUS DELIVERY
  deployed pipeline 2026.11.13 with 6 components

STAGE 4 — AUTOMATED TRIGGERING
 trigger  fired                                             reason
schedule   True 10000d since the last training run (threshold 30d)
new_data   True             7931 new labelled rows (threshold 500)



  trained v20261113T170950  roc_auc=0.7827  stage=staging


In [15]:
ORC.stage_cd_model(run_id, model_name=config.MODEL_NAME,
                   version=r4["model_version"].version, registry_mod=REG)
print()
print(REG.table(config.MODEL_NAME).to_string(index=False))

STAGE 5 — MODEL CONTINUOUS DELIVERY
  PROMOTED: roc_auc 0.7827 vs production 0.7827 (needs +0.0000)

         version      stage           algorithm  roc_auc  pr_auc  recall@top20%  rows             created
v20261113T170833   archived logistic_regression   0.7827  0.1769         0.4709  2548 2026-11-13T17:08:33
v20261113T170950 production logistic_regression   0.7827  0.1769         0.4709  2548 2026-11-13T17:09:50


Promotion is a **separate act** from training, and it is gated on beating the
incumbent. A pipeline that trains and deploys in one motion has nowhere to ask
"is this new model actually better than the one already serving?" — and
automatic retraining without that question is automatic degradation waiting for
a bad week of data.

The same six stages run unattended in `.github/workflows/`: `ci.yml` builds and
validates, `cd.yml` trains, promotes, monitors and publishes on a weekly
schedule.

In [16]:
import pathlib
for wf in sorted(pathlib.Path(PROJECT_DIR, ".github/workflows").glob("*.yml")):
    import yaml
    d = yaml.safe_load(wf.read_text())
    print(f"{wf.name}")
    for job, spec in d["jobs"].items():
        steps = len(spec.get("steps", []))
        print(f"   job '{job}'  {steps} steps  needs={spec.get('needs','-')}")

cd.yml
   job 'continuous-training'  8 steps  needs=-
ci.yml
   job 'code'  5 steps  needs=-
   job 'data-and-model'  6 steps  needs=code
   job 'package'  4 steps  needs=data-and-model


## Part 7 — Serving (slide 6), and the skew test

Slide 6 lists three deployment shapes; two are built here — **batch** scoring
for the dashboard and campaign, and an **online** predictor for one member at a
time.

Both read features through the feature store, so a served prediction uses the
same feature definitions the model was trained on. That is the property the
feature store exists to guarantee, which makes it worth testing rather than
assuming.

In [17]:
users = labels.user_id.drop_duplicates().tolist()
preds = SRV.batch_score(config.MODEL_NAME, F.ALL_FEATURE_SETS, users, run_id=run_id)
print(f"batch scored {len(preds):,} members")
print(preds.risk_band.value_counts().rename("members").to_frame().T.to_string())
print("\nhighest risk")
print(preds.nlargest(5,"churn_probability")[["user_id","churn_probability","risk_band"]]
      .round(4).to_string(index=False))

print("\nonline prediction — the REST-API shape from slide 6, without the HTTP")
print(" ", SRV.OnlinePredictor(config.MODEL_NAME, F.ALL_FEATURE_SETS).predict(users[0]))

print("\nTRAINING / SERVING SKEW")
print(" ", SRV.check_training_serving_skew(config.MODEL_NAME, F.ALL_FEATURE_SETS, labels))

batch scored 1,993 members
risk_band   low  medium  high
members    1594     299   100

highest risk
 user_id  churn_probability risk_band
     277             0.9998      high
      67             0.9988      high
     425             0.9935      high
     470             0.9901      high
     662             0.8263      high

online prediction — the REST-API shape from slide 6, without the HTTP
  {'user_id': 6, 'churn_probability': 0.0086, 'risk_band': 'low', 'model': 'lowrox_churn:v20261113T170950', 'served_at': '2026-11-13T17:09:50.695015+00:00'}

TRAINING / SERVING SKEW


  {'compared': 200, 'features': 37, 'mismatches': 0, 'verdict': 'training and serving agree', 'worst_features': {}}


Risk bands are **percentiles of the scored population**, not fixed probability
cuts. A retention team has a weekly capacity, not a probability threshold, and
a calibrated model on a 9% base rate rarely produces anything above 0.5 — fixed
cuts put everyone in "low" and the dashboard reports that there is nothing to
do. That was the first version's behaviour.

## Part 8 — Model drift (slides 13–19), each one simulated

Six checks. Rather than describe them, each is **fired on purpose**, the same
reasoning as the fault injection in the ETL pipeline: a monitor that has never
been shown to fire is a monitor nobody has tested.

Slide 14 asks for the distribution parameters to be stored at training time —
mean, median, standard deviation, IQR, kurtosis — and warns that central
tendency and spread must be compared **together**, because two distributions
can share a mean and nothing else. That is implemented literally, then
supplemented with PSI and the KS test.

In [18]:
mv = REG.production(config.MODEL_NAME)
stats = REG.training_stats(mv.name, mv.version)
print("TRAINING-TIME DISTRIBUTION, STORED WITH THE MODEL  (slide 14)")
print(stats.head(6).round(3).to_string())

TRAINING-TIME DISTRIBUTION, STORED WITH THE MODEL  (slide 14)
                   mean  median     std    iqr    p01      p99  kurtosis   skew  missing_rate
tenure_days      67.843  62.000  35.935  53.00  7.470  154.000    -0.521  0.501             0
log_tenure_days   4.067   4.143   0.627   0.83  2.135    5.043     1.475 -0.948             0
age_band          1.083   1.000   0.928   2.00  0.000    3.000    -0.543  0.544             0
gender_code       0.459   0.000   0.543   1.00  0.000    2.000    -0.799  0.598             0
was_referred      0.400   0.000   0.490   1.00  0.000    1.000    -1.833  0.409             0
area_hash         7.266   6.000   3.447   3.00  0.000   15.000     0.136  0.373             0


In [19]:
from sklearn.metrics import roc_auc_score
model = REG.load(mv.name, mv.version)
X_tr, y_tr = result["splits"].X_train, result["splits"].y_train
holdout = X_tr.sample(frac=0.4, random_state=7); y_hold = y_tr.loc[holdout.index]

def auc_of(X, y):
    return float(roc_auc_score(y, model.predict_proba(X)[:,1])) if y.nunique()>1 else None

# The concept-drift reference must be measured on the SAME population the
# live data is drawn from. Using the model's test-period ROC-AUC as the
# reference and comparing it to a training-period holdout reported a 0.115
# "drop" on untouched data — the training months carry a 13-14% churn rate
# and the test months 7-8%, so the two scores describe different populations,
# not model decay. Measured here instead of assumed.
REFERENCE_AUC = auc_of(holdout, y_hold)
print(f"reference ROC-AUC on an untouched holdout: {REFERENCE_AUC:.4f}")
print(f"(the model's test-period figure is {mv.metrics['roc_auc']:.4f} — a")
print(" different population, and not a valid baseline for this comparison)\n")

rows = []
for name, why in SIM.SCENARIOS.items():
    X, y = holdout.copy(), y_hold.copy()
    if name == "distribution_shift":
        X = SIM.shift_distribution(X, ["km_30d","sessions_30d","consistency_30d"], 2.0, stats)
    elif name == "spread_only":
        X = SIM.widen_spread(X, ["km_30d","pace_trend"], 3.0)
    elif name == "label_drift":
        y = SIM.shift_labels(y, 3.0)
    elif name == "concept_drift":
        y = SIM.break_relationship(X, y, 0.9)
    elif name == "edge_cases":
        X = SIM.inject_edge_cases(X, ["km_30d","sessions_30d"], 0.06, 40)
        X = SIM.inject_multivariate_edge_cases(X, ("km_30d","consistency_30d"), 0.05)
    elif name == "missing_as_feature":
        X = SIM.drop_feature_values(X, ["open_rate_30d","km_30d"], 0.5)
    rep = MON.monitor(training_stats=stats, training_sample=X_tr,
                      training_positive_rate=float(y_tr.mean()),
                      training_auc=REFERENCE_AUC,
                      live_features=X, live_labels=y, live_auc=auc_of(X, y),
                      feature_names=mv.features, verbose=False)
    sev = {f.check.split(" (")[0]: f.severity for f in rep["findings"]}
    rows.append({"scenario": name, **sev, "retrain": rep["retrain_triggered"]})

print("MONITORING RESPONSE BY SCENARIO")
print(pd.DataFrame(rows).to_string(index=False))

reference ROC-AUC on an untouched holdout: 0.6680
(the model's test-period figure is 0.7827 — a
 different population, and not a valid baseline for this comparison)



MONITORING RESPONSE BY SCENARIO
          scenario data distribution shift label drift concept drift edge cases missing-as-a-feature feedback loop  retrain
           healthy                      ok          ok            ok         ok                   ok       warning    False
distribution_shift                   alert          ok            ok    warning                   ok       warning     True
       spread_only                 warning          ok            ok      alert                   ok       warning     True
       label_drift                      ok       alert         alert         ok                   ok       warning     True
     concept_drift                      ok          ok         alert         ok                   ok       warning     True
        edge_cases                   alert          ok            ok    warning                   ok       warning     True
missing_as_feature                      ok          ok            ok         ok                alert

Read the matrix by row. The `healthy` row is quiet, which is the hard part —
a monitor that always alerts is a monitor that gets ignored. Each injected
scenario fires its own check and little else.

The `concept_drift` row is the one worth pointing at: **data distribution shift
reports `ok` while concept drift alerts.** That is slide 16's distinction being
detected rather than asserted — p(y|X) moved while p(X) stood still. The two
have different responses, so telling them apart matters.

Getting the healthy row quiet required two corrections, both of which are
ordinary mistakes:

- **Features that trend by construction.** Tenure is larger every month;
  lifetime counters only ever increase. Comparing their *level* against a
  training window that ended months ago reports drift on every run for ever.
  They are now reported as `expected_trend` — their *shape* is still checked.
- **A threshold set at exactly the rate that occurs by definition.** The
  p01–p99 band excludes 2% of training data by construction, so a 2% edge-case
  threshold warns on data drawn from the training distribution itself.
- **A baseline measured on a different population.** The first version used the
  model's test-period ROC-AUC as the concept-drift reference and compared it to
  a training-period holdout. That reported a 0.115 drop on untouched data: the
  training months carry a 13–14% churn rate and the test months 7–8%, so the
  two numbers describe different populations. A drift baseline has to be
  measured on data comparable to what is being monitored, or the monitor
  reports population differences as model decay.

In [20]:
live_now = FS.get_training_data(labels[labels.ts == labels.ts.max()], F.ALL_FEATURE_SETS)
monitoring = ORC.stage_monitoring(run_id, monitor_fn=MON.monitor,
    training_stats=stats, training_sample=X_tr,
    training_positive_rate=float(labels.is_churn.mean()),
    training_auc=mv.metrics["roc_auc"], live_features=live_now,
    live_labels=live_now["is_churn"], live_auc=None,
    feature_names=mv.features, verbose=True)

MONITORING REPORT
                             check severity                                                                                                 detail
data distribution shift (slide 14)  warning                          28 feature(s) moved: month_cos, month_sin, subscription_age_days, tenure_days
            label drift (slide 15)       ok                                                                churn rate 9.1% -> 6.6% (+27% relative)
          concept drift (slide 16)       ok                                                           no labelled outcomes yet for the live window
             edge cases (slide 17)  warning     2 feature(s) with >5% of values outside the training range; 1.8% of rows are multivariate outliers
   missing-as-a-feature (slide 19)       ok                                                                                  missingness unchanged
          feedback loop (slide 18)  warning features influenced by the model's own actions are in us

## Part 9 — The dashboard

One self-contained HTML file: no CDN, no build step, no network. It opens from
the Colab file browser, from a laptop, or from a USB stick — which matters for
a deliverable that has to still work when it is marked.

The numbers come from the Gold layer and the prediction table, not from
re-running anything. The dashboard and the retention campaign read the same
scored table, so they cannot disagree about who is at risk.

In [21]:
out = DASH.build(predictions=preds, model_version=mv, monitoring=monitoring,
                 feature_catalogue=FS.REGISTRY.catalogue(), labels=labels,
                 registry_table=REG.table(mv.name), lineage=ORC.lineage(run_id),
                 out_path=str(config.DASHBOARD))
print(f"written to {out}  ({os.path.getsize(out)/1024:.0f} KB)")

if IN_COLAB:
    from IPython.display import IFrame, display, HTML
    display(HTML(open(out).read()))
else:
    print("\n(open the file to view it)")

written to /home/claude/mlops_store_nb/lowrox_dashboard.html  (14 KB)

(open the file to view it)


In [22]:
print("ML METADATA STORE — the lineage of this run")
print(ORC.lineage(run_id).to_string(index=False))

ML METADATA STORE — the lineage of this run
                stage status                               at                                              detail         artifact
         1_experiment     ok 2026-11-13T17:09:50.022569+00:00                            best=logistic_regression              NaN
        3_cd_pipeline     ok 2026-11-13T17:09:50.036184+00:00                                                 NaN       2026.11.13
4_continuous_training     ok 2026-11-13T17:09:50.527804+00:00                                      roc_auc=0.7827 v20261113T170950
           5_cd_model     ok 2026-11-13T17:09:50.538121+00:00 roc_auc 0.7827 vs production 0.7827 (needs +0.0000) v20261113T170950
         6_monitoring     ok 2026-11-13T17:09:52.871839+00:00                                           no alerts              NaN


## Part 10 — Summary

In [23]:
summary = pd.DataFrame([
 {"chapter 7 topic":"DevOps vs MLOps: CI on data, schemas, models","evidence":f"{len(ci_report['results'])} CI checks across 4 suites"},
 {"chapter 7 topic":"Eight data science steps","evidence":"extraction through monitoring, Parts 3-8"},
 {"chapter 7 topic":"MLOps level 2","evidence":"6 stages + 2 GitHub Actions workflows"},
 {"chapter 7 topic":"Feature store: index + time index + features","evidence":f"{len(F.ALL_FEATURE_SETS)} feature sets, {sum(len(fs.features) for fs in F.ALL_FEATURE_SETS)} features"},
 {"chapter 7 topic":"Point-in-time correct join","evidence":"merge_asof backward + leakage assertion"},
 {"chapter 7 topic":"Offline / online consistency","evidence":"training-serving skew test: 0 mismatches"},
 {"chapter 7 topic":"Feature engineering","evidence":"12 techniques from slides 26-38"},
 {"chapter 7 topic":"Model validation against a baseline","evidence":"5 gates incl. 2 leakage sentinels"},
 {"chapter 7 topic":"Model registry and gated promotion","evidence":"staging -> production only if it beats the incumbent"},
 {"chapter 7 topic":"Data / label / concept drift","evidence":"7 scenarios simulated, each firing its own check"},
 {"chapter 7 topic":"Edge cases, feedback loops, missing-as-a-feature","evidence":"uni- and multi-dimensional, declared, monitored"},
 {"chapter 7 topic":"Dashboard","evidence":"self-contained HTML from Gold + predictions"},
])
print(summary.to_string(index=False))

                                 chapter 7 topic                                             evidence
    DevOps vs MLOps: CI on data, schemas, models                         25 CI checks across 4 suites
                        Eight data science steps             extraction through monitoring, Parts 3-8
                                   MLOps level 2                6 stages + 2 GitHub Actions workflows
    Feature store: index + time index + features                          4 feature sets, 41 features
                      Point-in-time correct join              merge_asof backward + leakage assertion
                    Offline / online consistency             training-serving skew test: 0 mismatches
                             Feature engineering                      12 techniques from slides 26-38
             Model validation against a baseline                    5 gates incl. 2 leakage sentinels
              Model registry and gated promotion staging -> production only if it 

### What to put in the report as limitations

Four, stated before anyone asks:

1. **The data is synthetic.** The pipeline is proved correct; the relationships
   it surfaces were placed in `calibration.py`. No commercial conclusion
   follows from the churn numbers.
2. **Monitoring has never seen production traffic.** The drift checks are
   exercised against held-out and deliberately corrupted data. That proves the
   detectors work; it does not prove the thresholds are right for real
   behaviour, which only live data can settle.
3. **The model is honest but not strong.** ROC-AUC around 0.75–0.79, lift about
   2.2× at the top 20%. That is a usable retention list and nothing more. The
   three suspiciously excellent versions that preceded it were all leakage.
4. **One label definition.** Churn is "cancelled within 60 days of the
   snapshot". A different horizon is a different problem, and the model should
   not be described as predicting churn in general.

### The thing most worth saying about this build

Three separate times, a model scored ROC-AUC near 0.995 and was worthless.
Every time the cause was the same shape: a **mutable column read at a past
snapshot**, where the application rewrites that column as part of the event
being predicted. The point-in-time join was correct on all three occasions —
the leak was in the feature definition.

The lesson is narrow and useful: *a column can only be a point-in-time feature
if its history exists.* Otherwise its present value is a message from the
future, and the only defence is a validation gate willing to reject a model for
being too good.